# Clinical NLP NER Exploration
This notebook explores dataset samples, label distribution, and qualitative model predictions.

In [ ]:
from collections import Counter
import pandas as pd
from datasets import load_dataset
from transformers import pipeline


In [ ]:
primary = 'bigbio/n2c2_2018_track2'
fallback = 'bigbio/bc5cdr'
try:
    ds = load_dataset(primary)
    dataset_used = primary
except Exception:
    ds = load_dataset(fallback)
    dataset_used = fallback
dataset_used

In [ ]:
sample_df = pd.DataFrame(ds['train'][:5])
sample_df[['tokens', 'ner_tags']].head()

In [ ]:
label_feature = ds['train'].features['ner_tags'].feature
label_names = label_feature.names if hasattr(label_feature, 'names') else None
counter = Counter()
for row in ds['train']:
    counter.update(row['ner_tags'])
if label_names:
    label_dist = pd.DataFrame({
        'label': [label_names[i] for i in counter.keys()],
        'count': list(counter.values())
    }).sort_values('count', ascending=False)
else:
    label_dist = pd.DataFrame({
        'label': list(counter.keys()),
        'count': list(counter.values())
    }).sort_values('count', ascending=False)
label_dist.head(20)

In [ ]:
# Requires a trained model in outputs/model
ner_pipe = pipeline('token-classification', model='outputs/model', tokenizer='outputs/model', aggregation_strategy='simple')
text = 'The patient with diabetes reports chest pain and is prescribed metformin after MRI.'
preds = ner_pipe(text)
preds

## Highlighted Entities
Use the output of the prediction cell above to inspect extracted diseases, drugs, symptoms, and procedures.